# Dimer generation, storage, and evaluation

## Import Modules for Setup

In [ ]:
# functions to instantiate modules
from orchestrator.utils.setup_input import setup_orch_modules

In [ ]:
# data standard defines a number of keys that are used throughout Orchestrator to access specific quantities
# it also includes common property maps that may be useful for many datasets
from orchestrator.utils.data_standard import (
    ENERGY_KEY,
    FORCES_KEY,
    METADATA_KEY,
    METADATA_PROPERTY_MAP,
    MOL_ID_KEY,
    SITE_TYPE_KEY,
    MOL_PROPERTY_MAP,
    MOL_PROPERTY_DEFINITION,
)

In [ ]:
inp = {
    'augmentor': {
        'augmentor_type': 'BASE',
        'augmentor_args': {},
    },
    'simulator': {
        'simulator_type': 'LAMMPS',
        'simulator_args': {
            'elements': ['C', 'H', 'O'],
            'code_path': '/PATH/TO/lmp'
        }
    },
    'storage': {
        'storage_type':'COLABFIT',
        'storage_args':{
            'credential_file':'PATH/TO/your_credentials.json'
        },
    },
    'scheduler': {
        'scheduler_type': 'SLURM',
        'scheduler_args': {
            'root_directory': './output',
            'queue': 'QUEUE_NAME',
            'account': 'ACCOUNT_NAME',
        }
    },
}

In [ ]:
(
    augmentor,
    _,
    _,
    _,
    _,
    simulator,
    storage,
    _,
    _,
    scheduler,
) = setup_orch_modules(inp)

## Inputs

In [ ]:
import numpy as np
# rotation vectors
z_rots = np.linspace(0, 345, 24).reshape(-1, 1)
mol1_rotations = np.hstack((np.zeros([z_rots.size, 2]), z_rots))
mol2_rotations = np.array([0, 0, 180])

# static translation vectors - 1D vector
mol1_translation = np.zeros(3)
mol2_translation = np.zeros(3)

# displacement vectors
radial_translation_dir = np.array([1, 0, 0])
radial_displacements = np.linspace(3, 13, 51)

# combined strucutre file defining the system to study
structure_file = 'inputs/lammps_data_2molecule.lmp'
# forcefield inputs
forcefield_file = 'inputs/lammps_forcefield_style.lmp'
include_files = [
    'inputs/lammps_parameters_covalent.lmp',
    'inputs/lammps_parameters_pairwise.lmp'
]

## Dimer generation

In [ ]:
dimers = augmentor.generate_dimer_configs(
    mol1_rotations,
    mol1_translation,
    mol2_rotations,
    mol2_translation,
    radial_translation_dir,
    radial_displacements,
    structure_file,
)

## ... with Storage

In [ ]:
dimers, ds_id = augmentor.generate_dimer_configs(
    mol1_rotations,
    mol1_translation,
    mol2_rotations,
    mol2_translation,
    radial_translation_dir,
    radial_displacements,
    structure_file,
    # add these optional arguments to the function call to automatically store the structures
    storage,
    'demo_dimer_generation',
)

## evaluate dimers with FF

#### NOTE: the dimers passed in here must be consistent if restarting - getting the dimers from storage can mix up the ordering and lead to incorrect results

In [ ]:
augmentor.evaluate_dimer_configs_simulator(
    dimers,
    simulator,
    scheduler,
    forcefield_file,
    structure_file,
    include_files,
    job_details={'walltime': 5},
)

output will be generated in `output/LAMMPSSimulator/dimer_scan/00000/energy_surface.png`